In [ ]:

### after running test1_copycopy.ipynb, test ViT in earnest 
### Case1 is use only one Fisher layer on vit_b_16

import torch, torchvision, time
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Dataset parameters
NUM_CLASSES = 10
BATCH_SIZE = 64
EPOCHS = 10
LR = 0.001

K_PROBES = 4
EPSILON = 0.1

# Device setup for CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load MNIST dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
 ])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

class MNISTClassifier(nn.Module):
    def __init__(self, num_classes, k_probes, epsilon):
        super().__init__()
        self.epsilon = epsilon
        self.k_probes = k_probes
        self.num_classes = num_classes

        v = torch.randn(k_probes, 2)
        v = v / torch.norm(v, dim=1, keepdim=True)
        self.register_buffer('probes', v)

        self.vit = torchvision.models.vit_b_16(weights=None)
        self.vit.conv_proj = nn.Conv2d(1, 768, kernel_size=16, stride=16, padding=0, bias=False)
        self.fisher_layer_indices = (6,)

        self.classifier = nn.Sequential(
            nn.Linear(768 + 3 + len(self.fisher_layer_indices), 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def compute_score(self, x):
        if not x.requires_grad:
            x.requires_grad_(True)

        energy = x.mean()
        grads = torch.autograd.grad(
            outputs=energy,
            inputs=x,
            create_graph=True,
            retain_graph=True
        )[0]

        grads_flat = grads.view(grads.size(0), -1)
        if grads_flat.size(1) >= 2:
            grads_2d = grads_flat[:, :2]
        else:
            grads_2d = grads_flat

        scores = -torch.matmul(grads_2d, self.probes.T)
        return scores

    def compute_local_fisher(self, x):
        scores = self.compute_score(x)
        fisher_info = (scores ** 2).mean(dim=1, keepdim=True)
        return fisher_info

    def forward(self, x):
        with torch.set_grad_enabled(True):
            if not x.requires_grad:
                x.requires_grad_(True)

            O0 = self.compute_local_fisher(x)

            perturbation = torch.randn_like(x) * self.epsilon
            I_pos = self.compute_local_fisher(x + perturbation)
            I_neg = self.compute_local_fisher(x - perturbation)

            O1 = (I_pos - I_neg) / (2 * self.epsilon)
            O2 = (I_pos - 2 * O0 + I_neg) / (self.epsilon ** 2)

        n = x.shape[0]
        h = self.vit._process_input(x)
        batch_class_token = self.vit.class_token.expand(n, -1, -1)
        h = torch.cat([batch_class_token, h], dim=1)
        h = h + self.vit.encoder.pos_embedding
        h = self.vit.encoder.dropout(h)

        fisher_terms = []
        for layer_idx, layer in enumerate(self.vit.encoder.layers):
            h = layer(h)
            if layer_idx in self.fisher_layer_indices:
                with torch.set_grad_enabled(True):
                    if not h.requires_grad:
                        h.requires_grad_(True)
                    fisher_terms.append(self.compute_local_fisher(h))

        h = self.vit.encoder.ln(h)
        cls_feature = h[:, 0]
        fish_concat = torch.cat(fisher_terms, dim=1)
        features = torch.cat([cls_feature, O0, O1, O2, fish_concat], dim=1)

        logits = self.classifier(features)
        return logits

def run_experiment(model, opt):
    model = model.to(device)
    optimizer = opt
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    test_accuracies = []

    print("Starting MNIST Training...")
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}/{len(train_loader)}: Loss={loss.item():.4f}")

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
        print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f} | Test Accuracy: {accuracy:.2f}%")

    print(f"Final Test Accuracy: {test_accuracies[-1]:.2f}%")
    return train_losses, test_accuracies

s = time.time()
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.Adam(model.parameters(), lr=LR)
train_losses_vit_b16_fisher_one_adam, test_accuracies_vit_b16_fisher_one_adam = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

s = time.time()
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.SGD(model.parameters(), lr=LR)
train_losses_vit_b16_fisher_one_sgd, test_accuracies_vit_b16_fisher_one_sgd = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

s = time.time() 
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.RMSprop(model.parameters(), lr=LR)
train_losses_vit_b16_fisher_one_rmsprop, test_accuracies_vit_b16_fisher_one_rmsprop = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

epochs = list(range(EPOCHS))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for loss, label, marker, color in [
    (train_losses_vit_b16_fisher_one_adam, 'Adam', 'o', 'blue'),
    (train_losses_vit_b16_fisher_one_sgd, 'SGD', 's', 'orange'),
    (train_losses_vit_b16_fisher_one_rmsprop, 'RMSprop', '^', 'red'),
]:
    ax1.plot(epochs, loss, marker=marker, linewidth=2, markersize=6, label=label, color=color)

for acc, label, marker, color in [
    (test_accuracies_vit_b16_fisher_one_adam, 'Adam', 'o', 'blue'),
    (test_accuracies_vit_b16_fisher_one_sgd, 'SGD', 's', 'orange'),
    (test_accuracies_vit_b16_fisher_one_rmsprop, 'RMSprop', '^', 'red'),
]:
    ax2.plot(epochs, acc, marker=marker, linewidth=2, markersize=6, label=label, color=color)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('MNIST + ViT_b_16_Fisher Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('MNIST + ViT_b_16_Fisher Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mnist_vit_b_16_fisher_training.png', dpi=150, bbox_inches='tight')
print("Training plots saved as 'mnist_vit_b_16_fisher_training.png'")
plt.show()


Using device: cuda
Train samples: 60000, Test samples: 10000
Starting MNIST Training...
Epoch 0, Batch 0/938: Loss=2.3546
Epoch 0, Batch 100/938: Loss=2.3021
Epoch 0, Batch 200/938: Loss=2.2997
Epoch 0, Batch 300/938: Loss=2.2952
Epoch 0, Batch 400/938: Loss=2.2945
Epoch 0, Batch 500/938: Loss=2.3136
Epoch 0, Batch 600/938: Loss=2.3029
Epoch 0, Batch 700/938: Loss=2.2937
Epoch 0, Batch 800/938: Loss=2.2953
Epoch 0, Batch 900/938: Loss=2.3052
Epoch 0 | Train Loss: 2.3038 | Test Accuracy: 11.35%
Epoch 1, Batch 0/938: Loss=2.3082
Epoch 1, Batch 100/938: Loss=2.3025
Epoch 1, Batch 200/938: Loss=2.2957
Epoch 1, Batch 300/938: Loss=2.3034
Epoch 1, Batch 400/938: Loss=2.3046
Epoch 1, Batch 500/938: Loss=2.3055
Epoch 1, Batch 600/938: Loss=2.2898
Epoch 1, Batch 700/938: Loss=2.2846
Epoch 1, Batch 800/938: Loss=2.3015
Epoch 1, Batch 900/938: Loss=2.3063
Epoch 1 | Train Loss: 2.3013 | Test Accuracy: 11.35%
Epoch 2, Batch 0/938: Loss=2.2926
Epoch 2, Batch 100/938: Loss=2.2998
Epoch 2, Batch 200/9

In [ ]:

### Case2 is use only one Fisher layer on vit_l_16

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Load MNIST dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
 ])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

class MNISTClassifier(nn.Module):
    def __init__(self, num_classes, k_probes, epsilon):
        super().__init__()
        self.epsilon = epsilon
        self.k_probes = k_probes
        self.num_classes = num_classes

        v = torch.randn(k_probes, 2)
        v = v / torch.norm(v, dim=1, keepdim=True)
        self.register_buffer('probes', v)

        self.vit = torchvision.models.vit_l_16(weights=None)
        self.vit.conv_proj = nn.Conv2d(1, 1024, kernel_size=16, stride=16, padding=0, bias=False)
        self.fisher_layer_indices = (12,)

        self.classifier = nn.Sequential(
            nn.Linear(1024 + 3 + len(self.fisher_layer_indices), 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def compute_score(self, x):
        if not x.requires_grad:
            x.requires_grad_(True)

        energy = x.mean()
        grads = torch.autograd.grad(
            outputs=energy,
            inputs=x,
            create_graph=True,
            retain_graph=True
        )[0]

        grads_flat = grads.view(grads.size(0), -1)
        if grads_flat.size(1) >= 2:
            grads_2d = grads_flat[:, :2]
        else:
            grads_2d = grads_flat

        scores = -torch.matmul(grads_2d, self.probes.T)
        return scores

    def compute_local_fisher(self, x):
        scores = self.compute_score(x)
        fisher_info = (scores ** 2).mean(dim=1, keepdim=True)
        return fisher_info

    def forward(self, x):
        with torch.set_grad_enabled(True):
            if not x.requires_grad:
                x.requires_grad_(True)

            O0 = self.compute_local_fisher(x)

            perturbation = torch.randn_like(x) * self.epsilon
            I_pos = self.compute_local_fisher(x + perturbation)
            I_neg = self.compute_local_fisher(x - perturbation)

            O1 = (I_pos - I_neg) / (2 * self.epsilon)
            O2 = (I_pos - 2 * O0 + I_neg) / (self.epsilon ** 2)

        n = x.shape[0]
        h = self.vit._process_input(x)
        batch_class_token = self.vit.class_token.expand(n, -1, -1)
        h = torch.cat([batch_class_token, h], dim=1)
        h = h + self.vit.encoder.pos_embedding
        h = self.vit.encoder.dropout(h)

        fisher_terms = []
        for layer_idx, layer in enumerate(self.vit.encoder.layers):
            h = layer(h)
            if layer_idx in self.fisher_layer_indices:
                with torch.set_grad_enabled(True):
                    if not h.requires_grad:
                        h.requires_grad_(True)
                    fisher_terms.append(self.compute_local_fisher(h))

        h = self.vit.encoder.ln(h)
        cls_feature = h[:, 0]
        fish_concat = torch.cat(fisher_terms, dim=1)
        features = torch.cat([cls_feature, O0, O1, O2, fish_concat], dim=1)

        logits = self.classifier(features)
        return logits

def run_experiment(model, opt):
    model = model.to(device)
    optimizer = opt
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    test_accuracies = []

    print("Starting MNIST Training...")
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}/{len(train_loader)}: Loss={loss.item():.4f}")

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
        print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f} | Test Accuracy: {accuracy:.2f}%")

    print(f"Final Test Accuracy: {test_accuracies[-1]:.2f}%")
    return train_losses, test_accuracies

s = time.time()
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.Adam(model.parameters(), lr=LR)
train_losses_vit_l16_fisher_one_adam, test_accuracies_vit_l16_fisher_one_adam = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

s = time.time()
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.SGD(model.parameters(), lr=LR)
train_losses_vit_l16_fisher_one_sgd, test_accuracies_vit_l16_fisher_one_sgd = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

s = time.time() 
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.RMSprop(model.parameters(), lr=LR)
train_losses_vit_l16_fisher_one_rmsprop, test_accuracies_vit_l16_fisher_one_rmsprop = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

epochs = list(range(EPOCHS))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for loss, label, marker, color in [
    (train_losses_vit_l16_fisher_one_adam, 'Adam', 'o', 'blue'),
    (train_losses_vit_l16_fisher_one_sgd, 'SGD', 's', 'orange'),
    (train_losses_vit_l16_fisher_one_rmsprop, 'RMSprop', '^', 'red'),
]:
    ax1.plot(epochs, loss, marker=marker, linewidth=2, markersize=6, label=label, color=color)

for acc, label, marker, color in [
    (test_accuracies_vit_l16_fisher_one_adam, 'Adam', 'o', 'blue'),
    (test_accuracies_vit_l16_fisher_one_sgd, 'SGD', 's', 'orange'),
    (test_accuracies_vit_l16_fisher_one_rmsprop, 'RMSprop', '^', 'red'),
]:
    ax2.plot(epochs, acc, marker=marker, linewidth=2, markersize=6, label=label, color=color)

ax1.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('MNIST + ViT_l_16_Fisher Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mnist_vit_l_16_fisher_training.png', dpi=150, bbox_inches='tight')
print("Training plots saved as 'mnist_vit_l_16_fisher_training.png'")
plt.show()



In [ ]:

### Case3 is use Fisher layer on all vit_b_16 layers

# Device setup for CUDA
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load MNIST dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
 ])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

class MNISTClassifier(nn.Module):
    def __init__(self, num_classes, k_probes, epsilon):
        super().__init__()
        self.epsilon = epsilon
        self.k_probes = k_probes
        self.num_classes = num_classes

        v = torch.randn(k_probes, 2)
        v = v / torch.norm(v, dim=1, keepdim=True)
        self.register_buffer('probes', v)

        self.vit = torchvision.models.vit_b_16(weights=None)
        self.vit.conv_proj = nn.Conv2d(1, 768, kernel_size=16, stride=16, padding=0, bias=False)
        self.fisher_layer_indices = (0,1,2,3,4,5,6,7,8,9,10,11)

        self.classifier = nn.Sequential(
            nn.Linear(768 + 3 + len(self.fisher_layer_indices), 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def compute_score(self, x):
        if not x.requires_grad:
            x.requires_grad_(True)

        energy = x.mean()
        grads = torch.autograd.grad(
            outputs=energy,
            inputs=x,
            create_graph=True,
            retain_graph=True
        )[0]

        grads_flat = grads.view(grads.size(0), -1)
        if grads_flat.size(1) >= 2:
            grads_2d = grads_flat[:, :2]
        else:
            grads_2d = grads_flat

        scores = -torch.matmul(grads_2d, self.probes.T)
        return scores

    def compute_local_fisher(self, x):
        scores = self.compute_score(x)
        fisher_info = (scores ** 2).mean(dim=1, keepdim=True)
        return fisher_info

    def forward(self, x):
        with torch.set_grad_enabled(True):
            if not x.requires_grad:
                x.requires_grad_(True)

            O0 = self.compute_local_fisher(x)

            perturbation = torch.randn_like(x) * self.epsilon
            I_pos = self.compute_local_fisher(x + perturbation)
            I_neg = self.compute_local_fisher(x - perturbation)

            O1 = (I_pos - I_neg) / (2 * self.epsilon)
            O2 = (I_pos - 2 * O0 + I_neg) / (self.epsilon ** 2)

        n = x.shape[0]
        h = self.vit._process_input(x)
        batch_class_token = self.vit.class_token.expand(n, -1, -1)
        h = torch.cat([batch_class_token, h], dim=1)
        h = h + self.vit.encoder.pos_embedding
        h = self.vit.encoder.dropout(h)

        fisher_terms = []
        for layer_idx, layer in enumerate(self.vit.encoder.layers):
            h = layer(h)
            if layer_idx in self.fisher_layer_indices:
                with torch.set_grad_enabled(True):
                    if not h.requires_grad:
                        h.requires_grad_(True)
                    fisher_terms.append(self.compute_local_fisher(h))

        h = self.vit.encoder.ln(h)
        cls_feature = h[:, 0]
        fish_concat = torch.cat(fisher_terms, dim=1)
        features = torch.cat([cls_feature, O0, O1, O2, fish_concat], dim=1)

        logits = self.classifier(features)
        return logits

def run_experiment(model, opt):
    model = model.to(device)
    optimizer = opt
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    test_accuracies = []

    print("Starting MNIST Training...")
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}/{len(train_loader)}: Loss={loss.item():.4f}")

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
        print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f} | Test Accuracy: {accuracy:.2f}%")

    print(f"Final Test Accuracy: {test_accuracies[-1]:.2f}%")
    return train_losses, test_accuracies

s = time.time()
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.Adam(model.parameters(), lr=LR)
train_losses_vit_b16_fisher_every_adam, test_accuracies_vit_b16_fisher_every_adam = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

s = time.time()
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.SGD(model.parameters(), lr=LR)
train_losses_vit_b16_fisher_every_sgd, test_accuracies_vit_b16_fisher_every_sgd = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

s = time.time() 
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.RMSprop(model.parameters(), lr=LR)
train_losses_vit_b16_fisher_every_rmsprop, test_accuracies_vit_b16_fisher_every_rmsprop = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

epochs = list(range(EPOCHS))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for loss, label, marker, color in [
    (train_losses_vit_b16_fisher_every_adam, 'Adam', 'o', 'blue'),
    (train_losses_vit_b16_fisher_every_sgd, 'SGD', 's', 'orange'),
    (train_losses_vit_b16_fisher_every_rmsprop, 'RMSprop', '^', 'red'),
]:
    ax1.plot(epochs, loss, marker=marker, linewidth=2, markersize=6, label=label, color=color)

for acc, label, marker, color in [
    (test_accuracies_vit_b16_fisher_every_adam, 'Adam', 'o', 'blue'),
    (test_accuracies_vit_b16_fisher_every_sgd, 'SGD', 's', 'orange'),
    (test_accuracies_vit_b16_fisher_every_rmsprop, 'RMSprop', '^', 'red'),
]:
    ax2.plot(epochs, acc, marker=marker, linewidth=2, markersize=6, label=label, color=color)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('MNIST + ViT_b_16_Fisher Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('MNIST + ViT_b_16_Fisher Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mnist_vit_b_16_fisher_training.png', dpi=150, bbox_inches='tight')
print("Training plots saved as 'mnist_vit_b_16_fisher_training.png'")
plt.show()



In [ ]:

### Case4 is use Fisher layer on all vit_l_16 layers

# Load MNIST dataset
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
 ])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")

class MNISTClassifier(nn.Module):
    def __init__(self, num_classes, k_probes, epsilon):
        super().__init__()
        self.epsilon = epsilon
        self.k_probes = k_probes
        self.num_classes = num_classes

        v = torch.randn(k_probes, 2)
        v = v / torch.norm(v, dim=1, keepdim=True)
        self.register_buffer('probes', v)

        self.vit = torchvision.models.vit_l_16(weights=None)
        self.vit.conv_proj = nn.Conv2d(1, 1024, kernel_size=16, stride=16, padding=0, bias=False)
        self.fisher_layer_indices = (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23)

        self.classifier = nn.Sequential(
            nn.Linear(1024 + 3 + len(self.fisher_layer_indices), 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def compute_score(self, x):
        if not x.requires_grad:
            x.requires_grad_(True)

        energy = x.mean()
        grads = torch.autograd.grad(
            outputs=energy,
            inputs=x,
            create_graph=True,
            retain_graph=True
        )[0]

        grads_flat = grads.view(grads.size(0), -1)
        if grads_flat.size(1) >= 2:
            grads_2d = grads_flat[:, :2]
        else:
            grads_2d = grads_flat

        scores = -torch.matmul(grads_2d, self.probes.T)
        return scores

    def compute_local_fisher(self, x):
        scores = self.compute_score(x)
        fisher_info = (scores ** 2).mean(dim=1, keepdim=True)
        return fisher_info

    def forward(self, x):
        with torch.set_grad_enabled(True):
            if not x.requires_grad:
                x.requires_grad_(True)

            O0 = self.compute_local_fisher(x)

            perturbation = torch.randn_like(x) * self.epsilon
            I_pos = self.compute_local_fisher(x + perturbation)
            I_neg = self.compute_local_fisher(x - perturbation)

            O1 = (I_pos - I_neg) / (2 * self.epsilon)
            O2 = (I_pos - 2 * O0 + I_neg) / (self.epsilon ** 2)

        n = x.shape[0]
        h = self.vit._process_input(x)
        batch_class_token = self.vit.class_token.expand(n, -1, -1)
        h = torch.cat([batch_class_token, h], dim=1)
        h = h + self.vit.encoder.pos_embedding
        h = self.vit.encoder.dropout(h)

        fisher_terms = []
        for layer_idx, layer in enumerate(self.vit.encoder.layers):
            h = layer(h)
            if layer_idx in self.fisher_layer_indices:
                with torch.set_grad_enabled(True):
                    if not h.requires_grad:
                        h.requires_grad_(True)
                    fisher_terms.append(self.compute_local_fisher(h))

        h = self.vit.encoder.ln(h)
        cls_feature = h[:, 0]
        fish_concat = torch.cat(fisher_terms, dim=1)
        features = torch.cat([cls_feature, O0, O1, O2, fish_concat], dim=1)

        logits = self.classifier(features)
        return logits

def run_experiment(model, opt):
    model = model.to(device)
    optimizer = opt
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    test_accuracies = []

    print("Starting MNIST Training...")
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0
        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

            if batch_idx % 100 == 0:
                print(f"Epoch {epoch}, Batch {batch_idx}/{len(train_loader)}: Loss={loss.item():.4f}")

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)

        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                _, predicted = torch.max(output.data, 1)
                total += target.size(0)
                correct += (predicted == target).sum().item()

        accuracy = 100 * correct / total
        test_accuracies.append(accuracy)
        print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f} | Test Accuracy: {accuracy:.2f}%")

    print(f"Final Test Accuracy: {test_accuracies[-1]:.2f}%")
    return train_losses, test_accuracies

s = time.time()
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.Adam(model.parameters(), lr=LR)
train_losses_vit_l16_fisher_every_adam, test_accuracies_vit_l16_fisher_every_adam = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

s = time.time()
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.SGD(model.parameters(), lr=LR)
train_losses_vit_l16_fisher_every_sgd, test_accuracies_vit_l16_fisher_every_sgd = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

s = time.time() 
model = MNISTClassifier(NUM_CLASSES, K_PROBES, EPSILON)
opt = optim.RMSprop(model.parameters(), lr=LR)
train_losses_vit_l16_fisher_every_rmsprop, test_accuracies_vit_l16_fisher_every_rmsprop = run_experiment(model, opt)
print(f"Total training time: {time.time() - s:.2f} seconds")

epochs = list(range(EPOCHS))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for loss, label, marker, color in [
    (train_losses_vit_l16_fisher_every_adam, 'Adam', 'o', 'blue'),
    (train_losses_vit_l16_fisher_every_sgd, 'SGD', 's', 'orange'),
    (train_losses_vit_l16_fisher_every_rmsprop, 'RMSprop', '^', 'red'),
]:
    ax1.plot(epochs, loss, marker=marker, linewidth=2, markersize=6, label=label, color=color)

for acc, label, marker, color in [
    (test_accuracies_vit_l16_fisher_every_adam, 'Adam', 'o', 'blue'),
    (test_accuracies_vit_l16_fisher_every_sgd, 'SGD', 's', 'orange'),
    (test_accuracies_vit_l16_fisher_every_rmsprop, 'RMSprop', '^', 'red'),
]:
    ax2.plot(epochs, acc, marker=marker, linewidth=2, markersize=6, label=label, color=color)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('MNIST + ViT_l_16_Fisher Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('MNIST + ViT_l_16_Fisher Test Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('mnist_vit_l_16_fisher_training.png', dpi=150, bbox_inches='tight')
print("Training plots saved as 'mnist_vit_l_16_fisher_training.png'")
plt.show()



In [ ]:

import pandas as pd

rows = []

g = globals()
loss_keys = sorted([k for k in g.keys() if k.startswith('train_losses_')])

for loss_key in loss_keys:
    suffix = loss_key.replace('train_losses_', '', 1)
    acc_key = f'test_accuracies_{suffix}'

    losses = g.get(loss_key)
    accuracies = g.get(acc_key)

    if losses is None or accuracies is None:
        continue

    n_epochs = min(len(losses), len(accuracies))
    for epoch in range(n_epochs):
        rows.append({
            'experiment': suffix,
            'epoch': epoch,
            'train_loss': float(losses[epoch]),
            'test_accuracy': float(accuracies[epoch])
        })

if not rows:
    raise RuntimeError(
        "No training outputs found. Run Cells 1–8 first so train_losses_* and test_accuracies_* variables exist."
    )

output_df = pd.DataFrame(rows).sort_values(['experiment', 'epoch']).reset_index(drop=True)
output_path = 'test11_output.csv'
output_df.to_csv(output_path, index=False)

print(f"Saved {len(output_df)} rows to {output_path}")
print(output_df.head())